# Notebook 01: Document Exploration & Ingestion Pipeline

**Hybrid-Agentic-RAG Capstone Demonstration**  
**Domain:** Technical Enterprise Documentation (Docker & Kubernetes Infrastructure)

This notebook demonstrates Phase 1 of the Capstone: Document Processing.
We explore multi-format document loading (HTML, PDF, DOCX), structure-aware text extraction, heading hierarchy extraction, and metadata preservation using the actual project ingestion modules in `src/ingestion/`.

## 1. Setup & Environment Configuration
We dynamically resolve the repository root so this notebook runs portably from any directory.

In [ ]:
import sys
from pathlib import Path

# Resolve repository root dynamically
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repository root: {repo_root}")
raw_docs_dir = repo_root / "data" / "raw"
print(f"Raw documents directory: {raw_docs_dir} (exists: {raw_docs_dir.exists()})")

## 2. Inspecting the Raw Document Collection
Let's list all documents in the knowledge base and categorize them by file extension.

In [ ]:
from collections import Counter

docs = list(raw_docs_dir.glob("*.*[a-zA-Z]"))
docs = [d for d in docs if not d.name.startswith(".")]

print(f"Total documents in collection: {len(docs)}\n")
ext_counts = Counter(d.suffix.lower() for d in docs)
for ext, count in ext_counts.items():
    print(f"  - {ext.upper():<8}: {count} document(s)")

print("\nDocument Manifest:")
for i, d in enumerate(sorted(docs, key=lambda p: p.name), 1):
    size_kb = d.stat().st_size / 1024
    print(f"  {i}. {d.name:<45} [{d.suffix.upper():<5}] ({size_kb:.1f} KB)")

## 3. Structure-Aware Document Loading
We use the project's `get_loader` factory to select the appropriate loader for each format (`HTMLLoader`, `PDFLoader`, `DocxLoader`).

In [ ]:
from src.ingestion.loaders.factory import get_loader

# 1. Load an HTML document (Docker bridge networking)
html_file = raw_docs_dir / "docker-bridge-network.html"
html_loader = get_loader(html_file)
html_doc = html_loader.load(html_file)

print(f"[HTML] File: {html_doc.filename}")
print(f"  - DocType: {html_doc.doc_type}")
print(f"  - Extracted structural elements: {len(html_doc.elements)}")
if html_doc.elements:
    sample_elem = html_doc.elements[0]
    print(f"  - Sample Heading: {sample_elem.heading}")
    print(f"  - Sample Section Path: {sample_elem.section_path}")
    print(f"  - Text snippet: {sample_elem.text[:140]}...")

In [ ]:
# 2. Load a PDF document (Kubernetes Pod Disruption Budget)
pdf_file = raw_docs_dir / "kubernetes-pod-disruption-budget.pdf"
if pdf_file.exists():
    pdf_loader = get_loader(pdf_file)
    pdf_doc = pdf_loader.load(pdf_file)
    print(f"[PDF] File: {pdf_doc.filename}")
    print(f"  - Extracted pages/elements: {len(pdf_doc.elements)}")
    for elem in pdf_doc.elements[:2]:
        print(f"    * Page {elem.page_number}: {elem.text[:100]}...")

In [ ]:
# 3. Load a DOCX document (Enterprise VPC Peering Guide)
docx_file = raw_docs_dir / "enterprise_vpc_guide.docx"
if docx_file.exists():
    docx_loader = get_loader(docx_file)
    docx_doc = docx_loader.load(docx_file)
    print(f"[DOCX] File: {docx_doc.filename}")
    print(f"  - Extracted paragraphs/tables: {len(docx_doc.elements)}")
    for elem in docx_doc.elements:
        print(f"    * Heading '{elem.heading}': {elem.text[:90]}...")

## 4. Document Statistics & Corpus Summary
Let's aggregate token and character statistics across the entire raw document corpus.

In [ ]:
import pandas as pd

stats_records = []
for doc_path in sorted(docs, key=lambda p: p.name):
    try:
        loader = get_loader(doc_path)
        loaded = loader.load(doc_path)
        total_chars = sum(len(e.text) for e in loaded.elements)
        total_words = sum(len(e.text.split()) for e in loaded.elements)
        approx_tokens = int(total_words * 1.33)
        headings_count = sum(1 for e in loaded.elements if e.heading)
        
        stats_records.append({
            "Filename": loaded.filename,
            "Format": loaded.doc_type.value,
            "Elements": len(loaded.elements),
            "Headings": headings_count,
            "Word Count": total_words,
            "Approx. Tokens": approx_tokens,
        })
    except Exception as e:
        print(f"Error processing {doc_path.name}: {e}")

df_stats = pd.DataFrame(stats_records)
print(df_stats.to_string(index=False))
print(f"\nTotal Corpus Tokens: ~{df_stats['Approx. Tokens'].sum():,}")

## 5. Key Architectural Insights

1. **Structure Preservation:** Naive text extraction loses markdown headings, document titles, and section boundaries. Our loaders build hierarchical `section_path` metadata that allows the retriever and LLM to know exactly which sub-heading context belongs to.
2. **Multi-Format Support:** The ingestion pipeline seamlessly handles HTML (with DOM stripping), PDF (with page indexing), and DOCX (with table and heading extraction).
3. **Next Step:** In `02_chunking_and_embeddings.ipynb`, we split these structural elements into 300–500 token chunks and generate vector embeddings.